# Weed Detection - GPU training on Google Colab

Fine-tunes **YOLO11s** on `Francesco/weed-crop-aerial` using Colab's free T4 GPU,
evaluates on the test split, and saves the model to your Google Drive **and**
downloads it.

### How to run
1. `Runtime` -> `Change runtime type` -> **T4 GPU** -> Save
2. `Runtime` -> **Run all**
3. **Keep this browser tab open and the laptop awake.** The whole run is
   ~20-30 min. Colab kills idle sessions and wipes everything, so don't walk
   away for long. Cell 3 mounts Google Drive, so even if it disconnects
   mid-training, `best.pt` is safe in Drive and you can just re-run.

Every cell is safe to re-run.

In [ ]:
# 1. GPU check
import torch, os
assert torch.cuda.is_available(), 'No GPU! Runtime > Change runtime type > T4 GPU, then Run all again.'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# 2. Get the project code + install (Colab already has a matching torch)
if not os.path.isdir('/content/repo'):
    !git clone --depth 1 https://github.com/Bhawna109/AI-based-Weed-detection.git /content/repo
%cd /content/repo
!pip -q install ultralytics datasets
import ultralytics; ultralytics.checks()

In [ ]:
# 3. Mount Google Drive - the trained model is written straight here, so a
#    Colab disconnect can't lose it. (A popup will ask you to allow access.)
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/weed_detection'
os.makedirs(DRIVE, exist_ok=True)
print('saving runs to:', DRIVE)

In [ ]:
# 4. Download + convert the dataset to YOLO format (writes configs/data.yaml)
%cd /content/repo
if not os.path.isdir('dataset/images/train'):
    !python src/fetch_hf_dataset.py --dataset Francesco/weed-crop-aerial
!python src/verify_dataset.py --samples 0

In [ ]:
# 5. TRAIN.  ~20-30 min for 60 epochs on a T4. Checkpoints stream to Drive
#    (--project). Change --model to yolo11m.pt for a bit more accuracy.
%cd /content/repo
!python src/train.py \
    --model yolo11s.pt \
    --epochs 60 \
    --batch 32 \
    --imgsz 640 \
    --device 0 \
    --workers 2 \
    --cache ram \
    --patience 20 \
    --project /content/drive/MyDrive/weed_detection/runs \
    --name weed_yolo11s_colab

In [ ]:
# 6. Evaluate on the untouched test split -> real Precision / Recall / mAP
%cd /content/repo
BEST = '/content/drive/MyDrive/weed_detection/runs/weed_yolo11s_colab/weights/best.pt'
assert os.path.exists(BEST), 'best.pt missing - let cell 5 (training) finish, then re-run.'
!python src/evaluate.py --weights "$BEST" --split test --name test_eval_colab

In [ ]:
# 7. Predict on the test images and show a few
%cd /content/repo
BEST = '/content/drive/MyDrive/weed_detection/runs/weed_yolo11s_colab/weights/best.pt'
!python src/predict.py --weights "$BEST" --source dataset/images/test --conf 0.25 --name colab_test
import glob, random
from IPython.display import Image, display
for p in random.sample(glob.glob('results/predictions/colab_test/*.jpg'), 6):
    display(Image(filename=p, width=500))

In [ ]:
# 8. Plots + copy everything worth keeping into Drive, then download a zip.
%cd /content/repo
!python src/plot_results.py --run /content/drive/MyDrive/weed_detection/runs/weed_yolo11s_colab

import shutil
out = '/content/drive/MyDrive/weed_detection/results'
os.makedirs(out, exist_ok=True)
for f in ['metrics_test.json', 'training_curves.png',
          'test_PR_curve.png', 'test_confusion_matrix_normalized.png']:
    if os.path.exists(f'results/{f}'):
        shutil.copy(f'results/{f}', f'{out}/{f}')

for p in ['results/training_curves.png',
          'results/test_confusion_matrix_normalized.png',
          'results/test_PR_curve.png']:
    try: display(Image(filename=p, width=650))
    except Exception: print('not found:', p)

shutil.make_archive('/content/weed_yolo11s_colab', 'zip', '/content/drive/MyDrive/weed_detection')
from google.colab import files
files.download('/content/weed_yolo11s_colab.zip')

## What you end up with

- **Google Drive** `MyDrive/weed_detection/` — the full run (`runs/weed_yolo11s_colab/weights/best.pt`) + `results/` plots and metrics. Survives disconnects.
- **A downloaded `weed_yolo11s_colab.zip`** with the same content.

### On your PC
Unzip and place:
- `runs/weed_yolo11s_colab/weights/best.pt` -> `results/runs/weed_yolo11s_colab/weights/best.pt`
- `results/*.png`, `results/metrics_test.json` -> `results/`

Then paste the metrics into the README Results table and commit. Predict locally:
```bash
python src/predict.py --weights results/runs/weed_yolo11s_colab/weights/best.pt --source <image_or_folder>
```